## Tool definition

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

<img src="./resources/tools.png" width="400" style="display:block; margin-left:0;">

## 工具 
#### 这个图可以看到大模型可以通过工具来扩大自己的能力，通过理理解内用户的问题, 产生调用工具和参数，调用工具，最后汇总结结果


### 我们先定一些基本的工具， 就是算术计算的函数， 例如加减乘除。

In [2]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [3]:
@tool("square_root")
def tool1(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [4]:
@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

In [5]:
tool1.invoke({"x": 467})

21.61018278497431

## 把工具赋能给agent

####  在agent 的基础之上， 增加工具，这样， agent 可以使用这些工具来扩展自己的能力。

In [6]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
import os
# model = init_chat_model("deepseek-chat")
# Alternative: local model via Ollama
# model = init_chat_model("qwen2.5:14b", model_provider="ollama")
model = init_chat_model(
    model="agnes-2.0-flash",
    model_provider="openai",
    base_url=os.getenv("AGNES_BASE_URL"),
    api_key=os.getenv("AGNES_API_KEY"),
)

agent = create_agent(
    model=model,
    tools=[tool1],
    system_prompt="你是一个算术巫师。使用你的工具来计算任何数字的平方根和平方。"
)

In [7]:
from langchain.messages import HumanMessage

question = HumanMessage(content="467的平方根是多少？")

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

467的平方根是21.61018278497431。


#### 注意agent 并不能直接使用工具，它是知道调用哪个工具，，并要把工具的参数赋给传给工具，工具去执行之后再交给agent 汇总。

In [8]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='467的平方根是多少？', additional_kwargs={}, response_metadata={}, id='becddd79-c087-4f28-b3e5-fb4cbc66e386'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 489, 'total_tokens': 517, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'agnes-2.0-flash', 'system_fingerprint': 'vllm-0.21.0-tp2-52bef988', 'id': 'chatcmpl-9a5450cd6bd36b6e', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ebc3b-98d2-7521-a101-79a46ce1cc62-0', tool_calls=[{'name': 'square_root', 'args': {'x': 467}, 'id': 'chatcmpl-tool-98fe55485649b476', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 489, 'output_tokens': 28, 'total_tokens': 517, 'input_token_details': {}, 'output_token_details': {}}),
 ToolMessage(content='21.61018278497431', name='square_root', id='05730bdb-00df-41f2-bb5c-bf1431578d74', tool_call_id=

In [9]:
print(response["messages"][1].tool_calls)

[{'name': 'square_root', 'args': {'x': 467}, 'id': 'chatcmpl-tool-98fe55485649b476', 'type': 'tool_call'}]
